In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("../../src")
import _util
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks, prepare_batch_multitoken_steering

In [4]:
import os
import glob

# Get all CSV files in the waterfall directory
waterfall_dir = "../../experiments/token_intervention/output/GPT-OSS_stepwise/waterfall"
csv_files = glob.glob(os.path.join(waterfall_dir, "*.csv"))

print(f"Found {len(csv_files)} CSV files in {waterfall_dir}")
print()

# Check row counts for each file
row_counts = {}
for csv_file in sorted(csv_files):
    try:
        df = pd.read_csv(csv_file)
        row_count = len(df)
        filename = os.path.basename(csv_file)
        row_counts[filename] = row_count
        print(f"{filename}: {row_count} rows")
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")

print()

# Check if all files have the same number of rows
if row_counts:
    unique_counts = set(row_counts.values())
    if len(unique_counts) == 1:
        print(f"✓ All files have the same number of rows: {list(unique_counts)[0]}")
    else:
        print(f"✗ Files have different row counts: {sorted(unique_counts)}")
        print("Files by row count:")
        for count in sorted(unique_counts):
            files_with_count = [f for f, c in row_counts.items() if c == count]
            print(f"  {count} rows: {files_with_count}")

Found 9 CSV files in ../../experiments/token_intervention/output/GPT-OSS_stepwise/waterfall

h.csv: 256 rows
h_pre_antipenultimate_sum.csv: 256 rows
h_pre_antipenultimate_sum_2.csv: 256 rows
h_pre_digit_2.csv: 256 rows
h_pre_final_sum.csv: 256 rows
h_pre_penultimate_sum.csv: 256 rows
h_pre_penultimate_sum_2.csv: 256 rows
h_pre_result.csv: 256 rows
h_pre_result_2.csv: 256 rows

✓ All files have the same number of rows: 256


In [5]:
# Check that base_1_num and base_2_num are consistent across all files for each row index
print("Checking consistency of base_1_num and base_2_num across files...")
print()

if not csv_files:
    print("No CSV files found to check.")
else:
    # Read all dataframes
    dataframes = {}
    for csv_file in sorted(csv_files):
        try:
            df = pd.read_csv(csv_file)
            filename = os.path.basename(csv_file)
            dataframes[filename] = df
        except Exception as e:
            print(f"Error reading {csv_file}: {e}")
            continue
    
    if not dataframes:
        print("No valid CSV files could be read.")
    else:
        # Get the first dataframe as reference
        reference_file = list(dataframes.keys())[0]
        reference_df = dataframes[reference_file]
        
        # Check if all files have base_1_num and base_2_num columns
        missing_columns = []
        for filename, df in dataframes.items():
            if 'base_1_num' not in df.columns or 'base_2_num' not in df.columns:
                missing_columns.append(filename)
        
        if missing_columns:
            print(f"✗ The following files are missing base_1_num or base_2_num columns:")
            for filename in missing_columns:
                print(f"  {filename}")
        else:
            # Check consistency row by row
            inconsistent_rows = []
            max_rows = min(len(df) for df in dataframes.values())
            
            for row_idx in range(max_rows):
                # Get base_1_num and base_2_num from reference file
                ref_base_1 = reference_df.iloc[row_idx]['base_1_num']
                ref_base_2 = reference_df.iloc[row_idx]['base_2_num']
                
                # Check against all other files
                for filename, df in dataframes.items():
                    if filename == reference_file:
                        continue
                    
                    current_base_1 = df.iloc[row_idx]['base_1_num']
                    current_base_2 = df.iloc[row_idx]['base_2_num']
                    
                    if current_base_1 != ref_base_1 or current_base_2 != ref_base_2:
                        inconsistent_rows.append({
                            'row_index': row_idx,
                            'reference_file': reference_file,
                            'ref_base_1': ref_base_1,
                            'ref_base_2': ref_base_2,
                            'inconsistent_file': filename,
                            'current_base_1': current_base_1,
                            'current_base_2': current_base_2
                        })
            
            if not inconsistent_rows:
                print(f"✓ base_1_num and base_2_num are consistent across all {len(dataframes)} files for all {max_rows} rows")
            else:
                print(f"✗ Found {len(inconsistent_rows)} inconsistencies:")
                for inconsistency in inconsistent_rows[:10]:  # Show first 10 inconsistencies
                    print(f"  Row {inconsistency['row_index']}: {inconsistency['reference_file']} has ({inconsistency['ref_base_1']}, {inconsistency['ref_base_2']}) but {inconsistency['inconsistent_file']} has ({inconsistency['current_base_1']}, {inconsistency['current_base_2']})")
                
                if len(inconsistent_rows) > 10:
                    print(f"  ... and {len(inconsistent_rows) - 10} more inconsistencies")

Checking consistency of base_1_num and base_2_num across files...

✓ base_1_num and base_2_num are consistent across all 9 files for all 256 rows


In [9]:
# Create a mapping from CSV files to their corresponding data files
data_dir = "../../data/GPT-OSS_stepwise"
csv_to_data_mapping = {}
missing_data_files = []

for csv_file in csv_files:
    # Extract just the filename from the full path
    filename = os.path.basename(csv_file)
    data_file_path = os.path.join(data_dir, f"h_prompts{filename[1:]}")
    
    if os.path.exists(data_file_path):
        csv_to_data_mapping[csv_file] = data_file_path
    else:
        missing_data_files.append(filename)

if missing_data_files:
    print(f"✗ The following files are missing from {data_dir}:")
    for filename in missing_data_files:
        print(f"  {filename}")
else:
    print(f"✓ All {len(csv_files)} files have corresponding files in {data_dir}")
    print(f"Created mapping for {len(csv_to_data_mapping)} file pairs")

✓ All 9 files have corresponding files in ../../data/GPT-OSS_stepwise
Created mapping for 9 file pairs


In [16]:
# Load data files and augment CSV files with counterfactual_output
augmented_dataframes = {}

for csv_file, data_file in csv_to_data_mapping.items():
    # Load CSV file
    csv_df = pd.read_csv(csv_file)
    
    # Load corresponding data file (also CSV)
    data_df = pd.read_csv(data_file)
    
    # Add counterfactual_output column to CSV dataframe
    csv_df['counterfactual_output'] = data_df['counterfactual_output'][:len(csv_df)]  # In case lengths don't match exactly
    
    augmented_dataframes[csv_file] = csv_df

print(f"✓ Augmented {len(augmented_dataframes)} dataframes with counterfactual_output")

# Check counterfactual_output presence in generated_text for each row across all files
faithful_count = 0

for i in range(256):
    all_files_faithful = True
    
    for csv_file, df in augmented_dataframes.items():
        if i >= len(df):
            all_files_faithful = False
            break
        
        counterfactual_output = df.iloc[i]['counterfactual_output']
        generated_text = df.iloc[i]['generated_text']
        
        if str(counterfactual_output) not in str(generated_text):
            all_files_faithful = False
            break
    
    if all_files_faithful:
        faithful_count += 1

print(f"✓ Number of rows (out of 256) where counterfactual_output is in generated_text for ALL files: {faithful_count}")

# Also show per-file statistics
print("\nPer-file statistics:")
for csv_file, df in augmented_dataframes.items():
    filename = os.path.basename(csv_file)
    file_faithful_count = 0
    
    for i in range(min(256, len(df))):
        counterfactual_output = df.iloc[i]['counterfactual_output']
        generated_text = df.iloc[i]['generated_text']
        
        if str(counterfactual_output) in str(generated_text):
            file_faithful_count += 1
    
    print(f"  {filename}: {file_faithful_count}/{min(256, len(df))} rows faithful")

✓ Augmented 9 dataframes with counterfactual_output
✓ Number of rows (out of 256) where counterfactual_output is in generated_text for ALL files: 131

Per-file statistics:
  h.csv: 256/256 rows faithful
  h_pre_antipenultimate_sum.csv: 256/256 rows faithful
  h_pre_antipenultimate_sum_2.csv: 256/256 rows faithful
  h_pre_digit_2.csv: 255/256 rows faithful
  h_pre_final_sum.csv: 256/256 rows faithful
  h_pre_penultimate_sum.csv: 256/256 rows faithful
  h_pre_penultimate_sum_2.csv: 256/256 rows faithful
  h_pre_result.csv: 131/256 rows faithful
  h_pre_result_2.csv: 256/256 rows faithful
